# 已訓練模型推理系統

本筆記本提供一個完整的推理系統，用於：
1. 選擇已保存的訓練好的模型
2. 從數據目錄中選擇測試數據集
3. 在新數據集上運行推理
4. 保存結果和評估指標

In [44]:
# =========================================
# 1. 環境設置和必要的導入
# =========================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import scipy.io as sio
import pandas as pd
import json
from pathlib import Path
from datetime import datetime
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# 設置設備
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用設備: {device}")
print(f"PyTorch 版本: {torch.__version__}")
print("="*60)

使用設備: cpu
PyTorch 版本: 2.5.1


In [45]:
# =========================================
# 2. GraphSAGE 模型定義
# =========================================

class GraphSAGE(nn.Module):
    """
    GraphSAGE 模型用於圖異常檢測
    """
    def __init__(self, in_feats, hidden=128, num_layers=3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_feats, hidden))
        for _ in range(1, num_layers):
            self.layers.append(nn.Linear(hidden * 2, hidden))
        self.classifier = nn.Linear(hidden, 2)  # 正常/異常
    
    def forward(self, x, edge_index, idx=None):
        h = x
        for i, layer in enumerate(self.layers):
            row, col = edge_index
            num_neigh = torch.bincount(row, minlength=h.size(0)).float()
            
            aggr = torch.zeros_like(h)
            aggr.index_add_(dim=0, index=row, source=h[col])
            aggr[row.unique()] /= (num_neigh[row.unique()] + 1e-10).unsqueeze(1)
            
            if i == 0:
                h = F.relu(layer(h))
            else:
                h = F.relu(layer(torch.cat([h, aggr], dim=1)))
        
        logits = self.classifier(h)
        return logits

In [46]:
# =========================================
# 3. 輔助函數：模型和數據集發現
# =========================================

def discover_saved_models(saved_models_dir="../saved_models"):
    """
    發現所有保存的模型文件
    返回: (模型列表, 最新模型路徑)
    """
    saved_models_dir = Path(saved_models_dir)
    if not saved_models_dir.exists():
        return [], None
    
    model_files = sorted(list(saved_models_dir.glob("*.pt")))
    return model_files, model_files[-1] if model_files else None


def discover_datasets(data_external_dir="../data/external"):
    """
    發現所有可用的数据集
    支持格式: .mat (MATLAB), .csv, .pt (PyTorch)
    返回: 字典 {'dataset_name': {'path': Path, 'type': 'mat'|'csv'|'pt', 'size': size}}
    """
    datasets = {}
    data_dir = Path(data_external_dir)
    
    if not data_dir.exists():
        return datasets
    
    # 搜索 MATLAB 文件
    for mat_file in data_dir.rglob("*.mat"):
        name = mat_file.stem
        datasets[name] = {
            'path': mat_file,
            'type': 'mat',
            'size': mat_file.stat().st_size / (1024**2)  # MB
        }
    
    # 搜索 CSV 文件
    for csv_file in data_dir.rglob("*.csv"):
        name = csv_file.stem
        datasets[name] = {
            'path': csv_file,
            'type': 'csv',
            'size': csv_file.stat().st_size / (1024**2)  # MB
        }
    
    # 搜索 PyTorch 文件
    for pt_file in data_dir.rglob("*.pt"):
        name = pt_file.stem
        datasets[name] = {
            'path': pt_file,
            'type': 'pt',
            'size': pt_file.stat().st_size / (1024**2)  # MB
        }
    
    return datasets


def print_available_options(saved_models, datasets):
    """打印可用的模型和數據集選項"""
    print("\n" + "="*60)
    print("📋 可用的已保存模型:")
    print("="*60)
    
    if saved_models:
        for i, model_path in enumerate(saved_models, 1):
            size = model_path.stat().st_size / (1024**2)
            print(f"  [{i}] {model_path.name} ({size:.2f} MB)")
    else:
        print("  ⚠️  未找到任何保存的模型")
    
    print("\n" + "="*60)
    print("📋 可用的數據集:")
    print("="*60)
    
    if datasets:
        for i, (name, info) in enumerate(datasets.items(), 1):
            print(f"  [{i}] {name} ({info['type'].upper()}) - {info['size']:.2f} MB")
            print(f"      路徑: {info['path']}")
    else:
        print("  ⚠️  未找到任何數據集")
    
    print("="*60 + "\n")

In [47]:
# =========================================
# 4. 發現並顯示可用的模型和數據集
# =========================================

# 發現模型和數據集
saved_models_dir = Path("../saved_models")
data_external_dir = Path("../data/external")

saved_models, latest_model = discover_saved_models(saved_models_dir)
available_datasets = discover_datasets(data_external_dir)

# 顯示可用選項
print_available_options(saved_models, available_datasets)


📋 可用的已保存模型:
  [1] graphsage_elliptic_20260321_164540.pt (0.34 MB)

📋 可用的數據集:
  [1] YelpChi (MAT) - 198.06 MB
      路徑: ../data/external/Yelp-Dataset/YelpChi.mat
  [2] yelpzip (CSV) - 391.83 MB
      路徑: ../data/external/Yelp-Dataset/yelpzip.csv



In [48]:
# =========================================
# 5. 選擇和加載模型
# =========================================

# ⚙️ 配置：修改這些變量來選擇模型和數據集
MODEL_INDEX = 0  # 選擇模型的索引（0 為第一個，-1 為最後一個）
DATASET_INDEX = 0  # 選擇數據集的索引（0 為第一個）

# 選擇模型
if not saved_models:
    print("❌ 錯誤：未找到任何保存的模型！")
else:
    selected_model_path = saved_models[MODEL_INDEX]
    print(f"\n{'='*60}")
    print(f"選擇的模型: {selected_model_path.name}")
    print(f"{'='*60}")
    
    try:
        # 加載模型檢查點
        checkpoint = torch.load(selected_model_path, map_location=device)
        
        # 提取模型架構信息
        model_arch = checkpoint['model_architecture']
        training_metrics = checkpoint.get('training_metrics', {})
        
        print(f"\n✓ 模型信息:")
        print(f"  架構: in_feats={model_arch['in_feats']}, hidden={model_arch['hidden']}, num_layers={model_arch['num_layers']}")
        print(f"  訓練性能:")
        print(f"    - Test AUC: {training_metrics.get('test_auc', 'N/A')}")
        print(f"    - Test AUPRC: {training_metrics.get('test_auprc', 'N/A')}")
        
        # 重建模型
        model = GraphSAGE(
            in_feats=model_arch['in_feats'],
            hidden=model_arch['hidden'],
            num_layers=model_arch['num_layers']
        ).to(device)
        
        # 加載模型權重
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()
        
        print(f"✓ 模型已加載到 {device}")
        
    except Exception as e:
        print(f"❌ 加載模型失敗: {e}")
        import traceback
        traceback.print_exc()


選擇的模型: graphsage_elliptic_20260321_164540.pt

✓ 模型信息:
  架構: in_feats=167, hidden=128, num_layers=3
  訓練性能:
    - Test AUC: 0.9864436705450649
    - Test AUPRC: 0.9413755738975966
✓ 模型已加載到 cpu


In [49]:
# =========================================
# 6. 數據加載函數（支持多種格式）
# =========================================

def load_mat_dataset(mat_path, feature_dim=None):
    """
    加載 MATLAB .mat 格式的數據集
    返回: (adj_matrix, labels, features) 或 None 如果加載失敗
    """
    try:
        mat_data = sio.loadmat(str(mat_path))
        
        # 嘗試找到鄰接矩陣和標籤
        adj_matrix = mat_data.get('net', mat_data.get('A', mat_data.get('adj', None)))
        labels = mat_data.get('label', mat_data.get('y', None))
        features = mat_data.get('features', mat_data.get('X', None))
        
        if labels is not None and len(labels.shape) > 1:
            labels = labels.flatten()
        
        print(f"\n✓ 加載 MATLAB 文件成功")
        print(f"  可用的鍵: {list(mat_data.keys())}")
        
        return adj_matrix, labels, features
    except Exception as e:
        print(f"❌ 加載 MATLAB 文件失敗: {e}")
        return None, None, None


def load_csv_dataset(csv_path):
    """
    加載 CSV 格式的數據集（通常為邊列表或特徵）
    返回: DataFrame 或 None 如果加載失敗
    """
    try:
        df = pd.read_csv(csv_path)
        print(f"\n✓ 加載 CSV 文件成功")
        print(f"  形狀: {df.shape}")
        print(f"  列: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"❌ 加載 CSV 文件失敗: {e}")
        return None


def load_pytorch_dataset(pt_path):
    """
    加載 PyTorch .pt 格式的數據集
    返回: 加載的數據字典 或 None 如果加載失敗
    """
    try:
        data = torch.load(pt_path, map_location=device)
        print(f"\n✓ 加載 PyTorch 文件成功")
        if isinstance(data, dict):
            print(f"  鍵: {list(data.keys())}")
        return data
    except Exception as e:
        print(f"❌ 加載 PyTorch 文件失敗: {e}")
        return None


def preprocess_adjacency_for_inference(adj_matrix, num_nodes=None, feature_dim=None):
    """
    將鄰接矩陣轉換為 edge_index 格式，並創建節點特徵
    返回: (features, edge_index, num_nodes_actual)
    """
    from scipy.sparse import csr_matrix
    
    # 轉換為稀疏矩陣
    if not isinstance(adj_matrix, csr_matrix):
        adj_matrix = csr_matrix(adj_matrix)
    
    num_nodes_actual = adj_matrix.shape[0]
    
    # 獲取邊
    row, col = adj_matrix.nonzero()
    edge_index = torch.tensor([row, col], dtype=torch.long, device=device)
    
    # 計算度數統計信息
    degree = torch.tensor(np.array(adj_matrix.sum(axis=1)).flatten(), 
                         dtype=torch.float, device=device)
    in_degree = torch.tensor(np.array(adj_matrix.sum(axis=0)).flatten(), 
                            dtype=torch.float, device=device)
    
    # 創建特徵矩陣
    if feature_dim is None:
        feature_dim = 168  # 默認特徵維度
    
    x = torch.randn(num_nodes_actual, feature_dim, device=device)
    x[:, 0] = degree / (degree.max() + 1e-8)
    if feature_dim > 1:
        x[:, 1] = in_degree / (in_degree.max() + 1e-8)
    
    return x, edge_index, num_nodes_actual

In [50]:
# =========================================
# 7. 選擇和加載數據集
# =========================================

if not available_datasets:
    print("\n❌ 錯誤：未找到任何數據集！")
else:
    print(f"\n{'='*60}")
    print(f"選擇數據集")
    print(f"{'='*60}")
    
    # 獲取數據集列表
    dataset_names = list(available_datasets.keys())
    selected_dataset_name = dataset_names[DATASET_INDEX]
    selected_dataset_info = available_datasets[selected_dataset_name]
    selected_dataset_path = selected_dataset_info['path']
    dataset_type = selected_dataset_info['type']
    
    print(f"\n選擇: {selected_dataset_name} ({dataset_type.upper()})")
    print(f"路徑: {selected_dataset_path}")
    
    # 根據數據集類型加載
    adj_matrix, labels, features = None, None, None
    
    if dataset_type == 'mat':
        mat_data_full = sio.loadmat(str(selected_dataset_path))
        print(f"\n✓ 加載 MATLAB 文件成功")
        print(f"  可用的鍵: {list(mat_data_full.keys())}")
        
        # 提取標籤
        labels = mat_data_full.get('label', None)
        if labels is not None and len(labels.shape) > 1:
            labels = labels.flatten()
        
        # 提取特徵
        features = mat_data_full.get('features', None)
        
        # 嘗試找到合適的網絡矩陣 (Yelp 數據有多個網絡)
        # 優先順序: 'homo' > 'net_rur' > 'net_rtr' > 'net_rsr' > 'net' > 'A'
        for key in ['homo', 'net_rur', 'net_rtr', 'net_rsr', 'net', 'A', 'adj']:
            if key in mat_data_full:
                adj_matrix = mat_data_full[key]
                print(f"  使用的網絡矩陣: '{key}'")
                break
        
        if adj_matrix is not None:
            print(f"  網絡大小: {adj_matrix.shape}")
        if features is not None:
            print(f"  特徵大小: {features.shape}")
        if labels is not None:
            print(f"  標籤: 正常={int((labels==0).sum())}, 異常={int((labels==1).sum())}")
            
    elif dataset_type == 'csv':
        df = load_csv_dataset(selected_dataset_path)
        if df is not None:
            print(f"\n  行數: {len(df)}")
            print(f"  前幾行:\n{df.head()}")
    elif dataset_type == 'pt':
        pt_data = load_pytorch_dataset(selected_dataset_path)
        if pt_data is not None and isinstance(pt_data, dict):
            if 'x' in pt_data and 'edge_index' in pt_data:
                adj_matrix = pt_data.get('adjacency_matrix', None)
                labels = pt_data.get('y', None)
                features = pt_data.get('x', None)


選擇數據集

選擇: YelpChi (MAT)
路徑: ../data/external/Yelp-Dataset/YelpChi.mat

✓ 加載 MATLAB 文件成功
  可用的鍵: ['__header__', '__version__', '__globals__', 'homo', 'net_rur', 'net_rtr', 'net_rsr', 'features', 'label']
  使用的網絡矩陣: 'homo'
  網絡大小: (45954, 45954)
  特徵大小: (45954, 32)
  標籤: 正常=39277, 異常=6677


In [51]:
# =========================================
# 8. 在選定的數據集上進行推理
# =========================================

if adj_matrix is not None:
    print(f"\\n{'='*60}")
    print(f"開始推理")
    print(f"{'='*60}")
    
    try:
        # 預處理數據
        x, edge_index, num_nodes = preprocess_adjacency_for_inference(
            adj_matrix, 
            feature_dim=model_arch['in_feats']
        )
        
        print(f"\\n✓ 數據預處理完成")
        print(f"  節點數: {num_nodes}")
        print(f"  特徵形狀: {x.shape}")
        print(f"  邊形狀: {edge_index.shape}")
        
        # 運行推理
        print(f"\\n⏳ 進行推理...")
        with torch.no_grad():
            logits = model(x, edge_index)
            probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
        
        print(f"✓ 推理完成")
        print(f"  預測概率範圍: [{probs.min():.4f}, {probs.max():.4f}]")
        print(f"  預測異常比例: {(probs > 0.5).sum() / len(probs) * 100:.1f}%")
        
        # 評估指標（如果有標籤）
        eval_results = {}
        if labels is not None:
            labels_clean = labels if isinstance(labels, np.ndarray) else np.array(labels).flatten()
            
            try:
                auc = roc_auc_score(labels_clean, probs)
                auprc = average_precision_score(labels_clean, probs)
                pred = (probs > 0.5).astype(int)
                tn, fp, fn, tp = confusion_matrix(labels_clean, pred).ravel()
                
                eval_results = {
                    'auc': float(auc),
                    'auprc': float(auprc),
                    'accuracy': float((pred == labels_clean).mean()),
                    'precision': float(precision_score(labels_clean, pred, zero_division=0)),
                    'recall': float(recall_score(labels_clean, pred, zero_division=0)),
                    'f1': float(f1_score(labels_clean, pred, zero_division=0)),
                    'confusion_matrix': {'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)},
                    'normal_count': int((labels_clean == 0).sum()),
                    'anomaly_count': int((labels_clean == 1).sum())
                }
                
                print(f"\\n📊 評估指標:")
                print(f"  AUC-ROC: {auc:.4f}")
                print(f"  AUPRC: {auprc:.4f}")
                print(f"  準確率: {(pred == labels_clean).mean():.4f}")
                print(f"  精確度: {precision_score(labels_clean, pred, zero_division=0):.4f}")
                print(f"  召回率: {recall_score(labels_clean, pred, zero_division=0):.4f}")
                print(f"  F1 分數: {f1_score(labels_clean, pred, zero_division=0):.4f}")
                print(f"  混淆矩陣: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
            except Exception as e:
                print(f"  ⚠️  無法計算評估指標: {e}")
        else:
            print(f"\\n  ⚠️  未提供標籤，無法計算評估指標")
        
    except Exception as e:
        print(f"❌ 推理失敗: {e}")
        import traceback
        traceback.print_exc()

\n============================================================
開始推理
\n✓ 數據預處理完成
  節點數: 45954
  特徵形狀: torch.Size([45954, 167])
  邊形狀: torch.Size([2, 7693958])
\n⏳ 進行推理...
✓ 推理完成
  預測概率範圍: [0.0000, 1.0000]
  預測異常比例: 2.2%
\n📊 評估指標:
  AUC-ROC: 0.5116
  AUPRC: 0.1498
  準確率: 0.8398
  精確度: 0.1650
  召回率: 0.0253
  F1 分數: 0.0439
  混淆矩陣: TN=38422, FP=855, FN=6508, TP=169


In [52]:
# =========================================
# 9. 生成詳細的文本報告函數
# =========================================

def generate_detailed_report(results_summary, probs, labels_clean=None, eval_results=None):
    """
    從 JSON 結果生成詳細的文本報告
    
    參數:
        results_summary: 推理結果摘要字典
        probs: 異常概率數組
        labels_clean: 真實標籤 (如果有)
        eval_results: 評估指標字典
    
    返回:
        報告字符串
    """
    report = []
    
    # 標題
    report.append("="*70)
    report.append("異常檢測模型推理詳細報告".center(70))
    report.append("="*70)
    report.append("")
    
    # 1. 推理元信息
    report.append("【推理元信息】")
    report.append("-" * 70)
    report.append(f"推理時間: {results_summary['inference_timestamp']}")
    report.append(f"模型文件: {results_summary['model_file']}")
    report.append(f"測試數據集: {results_summary['dataset_name']}")
    report.append(f"數據集格式: {results_summary['dataset_type'].upper()}")
    report.append("")
    
    # 2. 模型架構
    report.append("【模型架構】")
    report.append("-" * 70)
    model_arch = results_summary['model_architecture']
    report.append(f"輸入特徵維度: {model_arch['in_feats']}")
    report.append(f"隱藏層維度: {model_arch['hidden']}")
    report.append(f"層數: {model_arch['num_layers']}")
    report.append("")
    
    # 3. 原始訓練性能
    report.append("【原始訓練性能 (Elliptic 數據集)】")
    report.append("-" * 70)
    training_metrics = results_summary['original_training_metrics']
    if training_metrics:
        for key, value in training_metrics.items():
            report.append(f"{key}: {value}")
    else:
        report.append("無可用的訓練指標")
    report.append("")
    
    # 4. 測試數據集統計
    report.append("【測試數據集統計】")
    report.append("-" * 70)
    stats = results_summary['inference_statistics']
    report.append(f"節點總數: {stats['num_nodes']:,}")
    report.append(f"邊總數: {stats['num_edges']:,}")
    report.append(f"平均度數: {stats['num_edges'] * 2 / stats['num_nodes']:.2f}")
    
    if labels_clean is not None:
        num_normal = (labels_clean == 0).sum()
        num_anomaly = (labels_clean == 1).sum()
        report.append(f"正常節點數: {num_normal:,} ({num_normal/len(labels_clean)*100:.2f}%)")
        report.append(f"異常節點數: {num_anomaly:,} ({num_anomaly/len(labels_clean)*100:.2f}%)")
    report.append("")
    
    # 5. 預測概率統計
    report.append("【預測概率統計】")
    report.append("-" * 70)
    report.append(f"異常概率最小值: {stats['anomaly_probability_min']:.4f}")
    report.append(f"異常概率最大值: {stats['anomaly_probability_max']:.4f}")
    report.append(f"異常概率平均值: {stats['anomaly_probability_mean']:.4f}")
    report.append(f"異常概率標準差: {probs.std():.4f}")
    report.append(f"預測異常比例: {stats['predicted_anomaly_ratio']*100:.2f}%")
    report.append("")
    
    # 6. 評估指標 (如果有標籤)
    if eval_results and eval_results:
        report.append("【推理評估指標】")
        report.append("-" * 70)
        
        # 基本指標
        if 'auc' in eval_results:
            report.append(f"AUC-ROC: {eval_results['auc']:.4f}")
        if 'auprc' in eval_results:
            report.append(f"AUPRC: {eval_results['auprc']:.4f}")
        if 'accuracy' in eval_results:
            report.append(f"準確率 (Accuracy): {eval_results['accuracy']:.4f}")
        if 'precision' in eval_results:
            report.append(f"精確度 (Precision): {eval_results['precision']:.4f}")
        if 'recall' in eval_results:
            report.append(f"召回率 (Recall): {eval_results['recall']:.4f}")
        if 'f1' in eval_results:
            report.append(f"F1-分數: {eval_results['f1']:.4f}")
        
        # 混淆矩陣
        if 'confusion_matrix' in eval_results:
            report.append("")
            report.append("混淆矩陣:")
            cm = eval_results['confusion_matrix']
            tn = cm['tn']
            fp = cm['fp']
            fn = cm['fn']
            tp = cm['tp']
            report.append(f"  真陰性 (TN): {tn:,}")
            report.append(f"  假陽性 (FP): {fp:,}")
            report.append(f"  假陰性 (FN): {fn:,}")
            report.append(f"  真陽性 (TP): {tp:,}")
            
            # 詳細的分類統計
            report.append("")
            report.append("詳細分類統計:")
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
            report.append(f"  靈敏度 (Sensitivity): {sensitivity:.4f}")
            report.append(f"  特異度 (Specificity): {specificity:.4f}")
        
        report.append("")
    else:
        report.append("【評估指標】")
        report.append("-" * 70)
        report.append("未提供標籤，無法計算評估指標。")
        report.append("")
    
    # 7. 推理結果摘要
    report.append("【推理結果摘要】")
    report.append("-" * 70)
    report.append("本次推理在外部數據集上進行了異常檢測。")
    report.append(f"預測共有 {(probs > 0.5).sum():,} 個異常節點")
    report.append(f"占比 {(probs > 0.5).sum() / len(probs) * 100:.2f}%")
    report.append("")
    
    # 8. 文件說明
    report.append("【生成文件說明】")
    report.append("-" * 70)
    report.append("本文件夾包含以下文件:")
    report.append("  • inference_summary_*.json - 完整的推理結果和指標 (JSON 格式)")
    report.append("  • detailed_predictions_*.csv - 每個節點的詳細預測概率 (CSV 格式)")
    report.append("  • inference_report_*.txt - 本報告 (文本格式)")
    report.append("")
    
    # 9. 文件尾
    report.append("="*70)
    report.append(f"報告生成時間: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    report.append("="*70)
    
    return "\n".join(report)


# =========================================
# 9b. 保存推理結果到帶日期的文件夾
# =========================================

if 'probs' in locals():
    experiments_dir = Path("../experiments")
    experiments_dir.mkdir(exist_ok=True)
    
    # 生成時間戳
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 創建結果摘要
    results_summary = {
        'inference_timestamp': datetime.now().isoformat(),
        'model_file': selected_model_path.name,
        'dataset_name': selected_dataset_name,
        'dataset_type': dataset_type,
        'model_architecture': model_arch,
        'original_training_metrics': training_metrics,
        'inference_results': eval_results if eval_results else {'note': 'No labels provided'},
        'inference_statistics': {
            'num_nodes': int(num_nodes),
            'num_edges': int(edge_index.shape[1]),
            'anomaly_probability_min': float(probs.min()),
            'anomaly_probability_max': float(probs.max()),
            'anomaly_probability_mean': float(probs.mean()),
            'predicted_anomaly_ratio': float((probs > 0.5).sum() / len(probs))
        }
    }
    
    print(f"\n{'='*60}")
    print(f"保存推理結果")
    print(f"{'='*60}")
    
    # 創建具有描述性的文件夾名稱
    folder_name = f"{selected_dataset_name}_inference_{timestamp}"
    result_folder = experiments_dir / folder_name
    result_folder.mkdir(exist_ok=True)
    
    print(f"\n✓ 創建結果文件夾: {folder_name}")
    
    # 1. 保存 JSON 摘要
    json_path = result_folder / f"inference_summary_{timestamp}.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(results_summary, f, indent=2, ensure_ascii=False)
    print(f"✓ 摘要已保存: inference_summary_{timestamp}.json")
    
    # 2. 保存詳細預測 CSV
    csv_path = result_folder / f"detailed_predictions_{timestamp}.csv"
    predictions_df = pd.DataFrame({
        'node_id': range(len(probs)),
        'anomaly_probability': probs,
        'predicted_label': (probs > 0.5).astype(int)
    })
    
    if labels is not None:
        labels_clean = labels if isinstance(labels, np.ndarray) else np.array(labels).flatten()
        predictions_df['true_label'] = labels_clean
    else:
        labels_clean = None
    
    predictions_df.to_csv(csv_path, index=False)
    print(f"✓ 預測詳情已保存: detailed_predictions_{timestamp}.csv")
    print(f"  - 共 {len(predictions_df):,} 行數據")
    
    # 3. 保存詳細文本報告 (基於 JSON 數據)
    report_path = result_folder / f"inference_report_{timestamp}.txt"
    report_content = generate_detailed_report(
        results_summary, 
        probs, 
        labels_clean=labels_clean,
        eval_results=eval_results if eval_results else None
    )
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(report_content)
    print(f"✓ 詳細報告已保存: inference_report_{timestamp}.txt")
    
    print(f"\n{'='*60}")
    print(f"✅ 推理完成！")
    print(f"結果文件夾: {result_folder.absolute()}")
    print(f"文件夾內容:")
    print(f"  • inference_summary_{timestamp}.json")
    print(f"  • detailed_predictions_{timestamp}.csv")
    print(f"  • inference_report_{timestamp}.txt")
    print(f"{'='*60}")
else:
    print(f"\n⚠️  未進行推理，無法保存結果")


保存推理結果

✓ 創建結果文件夾: YelpChi_inference_20260321_174009
✓ 摘要已保存: inference_summary_20260321_174009.json
✓ 預測詳情已保存: detailed_predictions_20260321_174009.csv
  - 共 45,954 行數據
✓ 詳細報告已保存: inference_report_20260321_174009.txt

✅ 推理完成！
結果文件夾: /Users/mankin/Desktop/Anomaly_Detection_on_Graph/notebooks/../experiments/YelpChi_inference_20260321_174009
文件夾內容:
  • inference_summary_20260321_174009.json
  • detailed_predictions_20260321_174009.csv
  • inference_report_20260321_174009.txt


## 使用說明

### 快速開始

1. **第4塊**: 運行此單元格以發現所有可用的模型和數據集
2. **第5塊**: 配置 `MODEL_INDEX` 和 `DATASET_INDEX` 變量來選擇要使用的模型和數據集，然後運行
3. **第7塊**: 運行此單元格以加載選定的數據集
4. **第8塊**: 運行此單元格以進行推理
5. **第9塊**: 運行此單元格以保存結果

### 配置變量

在第5塊中修改以下變量：
- `MODEL_INDEX`: 整數索引，0 表示第一個模型，-1 表示最後一個（最新的）模型
- `DATASET_INDEX`: 整數索引，0 表示第一個數據集

### 支持的數據格式

- **MATLAB (.mat)**: 支持 `net` 或 `A` 作為鄰接矩陣，`label` 作為標籤
- **CSV (.csv)**: 邊列表或特徵矩陣格式
- **PyTorch (.pt)**: 包含 `x`, `edge_index`, `y` 等字段的字典

### 輸出結果

推理完成後，結果將保存到 `../experiments/` 目錄：
- `inference_summary_*.json`: 完整的推理摘要和評估指標
- `detailed_predictions_*.csv`: 每個節點的詳細預測結果
- `inference_report_*.txt`: 人類可讀的推理報告